In [2]:
import requests
import pandas as pd
import datetime
import pytz

In [ ]:
from datetime import timedelta
def subtract_24h_iso(iso_time: str) -> str:
    dt = datetime.fromisoformat(iso_time.replace("Z", "+00:00"))
    
    prev_dt = dt - timedelta(hours=24)

    return prev_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
import requests
from dateutil import parser
from dateutil.tz import gettz
from datetime import datetime, timezone, timedelta 

def get_historical_markets_for_timeframe():
    start_date = datetime(2025, 2, 1, tzinfo=timezone.utc)
    end_date = datetime(2026, 2, 28, 23, 59, 59, tzinfo=timezone.utc)
    # updated to historical, pretty similar to live tho
    historical_url = "https://external-api.kalshi.com/trade-api/v2/historical/markets"
    
    all_historical_markets = []
    cursor = None
    
    while True:
        params = {
            "series_ticker": "KXMLBGAME",
            "limit": 1000
        }
        if cursor:
            params['cursor'] = cursor
            
        response = requests.get(historical_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        markets = data.get("markets", [])
        
        for market in markets:
            close_time_str = market.get('close_time') 
            if close_time_str:
                close_dt = parser.parse(close_time_str)
                if start_date <= close_dt <= end_date:
                    all_historical_markets.append(market)
                    
        cursor = data.get("cursor")
        if not cursor:
            break 
            
    print(f"Found {len(all_historical_markets)} historical games.")
    return all_historical_markets

def transform_markets(all_markets):
    new_markets = []
    tzinfos = {
        "EDT": gettz("US/Eastern"), "EST": gettz("US/Eastern"), 
        "PDT": gettz("US/Pacific"), "PST": gettz("US/Pacific"),
        "CDT": gettz("US/Central"), "CST": gettz("US/Central"),
        "MDT": gettz("US/Mountain"), "MST": gettz("US/Mountain")
    }
               
    for market in all_markets:
        rules_primary = market.get('rules_primary', '')
        ticker = market.get('ticker', '')
        close_time = market.get('close_time')
        
        try:
            dt_start = None
            
            try:
                time_str = rules_primary.replace(" at ", " ")
                dt = parser.parse(time_str, tzinfos=tzinfos)
                dt_start = dt.replace(minute=0, second=0, microsecond=0) - timedelta(hours=1)
            except Exception:
                if close_time:
                    dt = parser.parse(close_time)
                    dt_start = dt.replace(minute=0, second=0, microsecond=0) - timedelta(hours=8)

            # probably dont need once sleep changed
            if dt_start is None:
                print(f"Skipping {ticker} - No valid time or close_time found.")
                continue
            
            start_ts = int(dt_start.timestamp())
            
            # Calculate end_ts (8 hours after the start_ts)
            dt_end = dt_start + timedelta(hours=8)
            end_ts = int(dt_end.timestamp())
            
            new_dict = {
                'start_ts': start_ts,
                'end_ts': end_ts,
                'series_ticker': 'KXMLBGAME',
                'market_ticker': ticker,
                'period_interval': 60,
                'include_latest_before_start': "true"
            }
            
            new_markets.append(new_dict)
            
        except Exception as e:
            print(f"Critical error processing {ticker}: {e}")
            
    return new_markets

In [9]:
historical_mkts = get_historical_markets_for_timeframe()

Found 4414 historical games.


In [12]:
new_markets = transform_markets(historical_mkts)

In [14]:
subset = new_markets[:3]
subset

[{'start_ts': 1761937200,
  'end_ts': 1761966000,
  'series_ticker': 'KXMLBGAME',
  'market_ticker': 'KXMLBGAME-25OCT31LADTOR-TOR',
  'period_interval': 60,
  'include_latest_before_start': 'true'},
 {'start_ts': 1761937200,
  'end_ts': 1761966000,
  'series_ticker': 'KXMLBGAME',
  'market_ticker': 'KXMLBGAME-25OCT31LADTOR-LAD',
  'period_interval': 60,
  'include_latest_before_start': 'true'},
 {'start_ts': 1761764400,
  'end_ts': 1761793200,
  'series_ticker': 'KXMLBGAME',
  'market_ticker': 'KXMLBGAME-25OCT29TORLAD-TOR',
  'period_interval': 60,
  'include_latest_before_start': 'true'}]

In [ ]:
import requests
import pandas as pd
import time

def fetch_all_market_candlesticks_fully_unpacked(new_markets):
    master_candlesticks_list = []
    session = requests.Session()
    
    for market_params in new_markets:
        series_ticker = market_params['series_ticker']
        market_ticker = market_params['market_ticker']
        
        url = f"https://external-api.kalshi.com/trade-api/v2/historical/markets/{market_ticker}/candlesticks"
        
        params = {
            "start_ts": market_params['start_ts'],
            "end_ts": market_params['end_ts'],
            "period_interval": market_params['period_interval'],
            "include_latest_before_start": "true" 
        }
        
        max_retries = 5
        backoff_time = 1.0
        success = False
        
        while max_retries > 0:
            try:
                response = session.get(url, params=params)
                
                # checking for 429 rate limit error
                if response.status_code == 429:
                    print(f"Rate limited (429) on {market_ticker}. Backing off for {backoff_time}s...")
                    time.sleep(backoff_time)
                    max_retries -= 1
                    backoff_time *= 2  # Double the wait time for the next attempt
                    continue
                
                # raise others errors
                response.raise_for_status() 
                
                data = response.json()
                candles = data.get("candlesticks", [])
                
                for candle in candles:
                    candle['market_ticker'] = market_ticker
                    
                master_candlesticks_list.extend(candles)
                success = True
                break  # end loop upon success
                    
            except requests.exceptions.RequestException as e:
                print(f"Network error fetching data for {market_ticker}: {e}")
                break  # end retry loop for non 429 errors
        
        if not success:
            print(f"Skipping {market_ticker} after multiple failed attempts.")
            
        time.sleep(0.20) # i could probably lower this for faster compiling
        
    # flattening
    if not master_candlesticks_list:
        return pd.DataFrame()
        
    df = pd.json_normalize(master_candlesticks_list)
    
    # dollar -> float
    dollar_cols = [col for col in df.columns if 'dollars' in col]
    for col in dollar_cols:
        df[col] = df[col].astype(float)
        
    if 'market_ticker' in df.columns:
        cols = ['market_ticker'] + [col for col in df.columns if col != 'market_ticker']
        df = df[cols]
        
    return df

In [34]:
hist_data01 = fetch_all_market_candlesticks_fully_unpacked(subset)

In [ ]:
# Feb 2025 to Feb 2026 ie 2025 baseball season
hist_dataFinal = fetch_all_market_candlesticks_fully_unpacked(new_markets)

In [36]:
hist_dataFinal.to_csv("kalshiBaseballHistData.csv")

In [38]:
hist_dataFinal.head()

,market_ticker,end_period_ts,open_interest,volume,price.close,price.high,price.low,price.mean,price.open,price.previous,yes_ask.close,yes_ask.high,yes_ask.low,yes_ask.open,yes_bid.close,yes_bid.high,yes_bid.low,yes_bid.open
0,KXMLBGAME-25OCT31LADTOR-TOR,1761937200,1216969.00,75127.00,0.4400,0.4400,0.4300,0.4394,0.4400,0.4400,0.4400,0.4400,0.4400,0.4400,0.4300,0.4300,0.4300,0.4300
1,KXMLBGAME-25OCT31LADTOR-TOR,1761940800,1391665.00,179433.00,0.4400,0.4400,0.4300,0.4365,0.4400,0.4400,0.4400,0.4400,0.4400,0.4400,0.4300,0.4300,0.4300,0.4300
2,KXMLBGAME-25OCT31LADTOR-TOR,1761944400,1551208.00,161400.00,0.4400,0.4400,0.4300,0.4398,0.4400,0.4400,0.4400,0.4400,0.4400,0.4400,0.4300,0.4300,0.4300,0.4300
3,KXMLBGAME-25OCT31LADTOR-TOR,1761948000,1702257.00,151859.00,0.4400,0.4400,0.4300,0.4399,0.4400,0.4400,0.4400,0.4400,0.4400,0.4400,0.4300,0.4300,0.4300,0.4300
4,KXMLBGAME-25OCT31LADTOR-TOR,1761951600,1954515.00,255832.00,0.4400,0.4400,0.4300,0.4397,0.4400,0.4400,0.4400,0.4400,0.4400,0.4400,0.4300,0.4300,0.4300,0.4300


In [ ]:
def odds(p):
    if p >= 0.5:
        return round(-p / (1 - p) * 100)
    return round((1 - p) / p * 100)

In [43]:
odds(.44)

127

In [37]:
len(hist_dataFinal)

38531

## another way to fetch historical data

In [23]:
import requests
from dateutil import parser
from datetime import datetime, timezone

all_markets = []
cursor = None
base = "https://external-api.kalshi.com/trade-api/v2"

# Define your 1-year lookback window
start_date = datetime(2025, 2, 1, tzinfo=timezone.utc)
end_date = datetime(2026, 2, 28, 23, 59, 59, tzinfo=timezone.utc)

while True:
    params = {
        "series_ticker": "KXMLBGAME", 
        "limit": 1000  # Increased to 1000 (max) to make pagination much faster
        # 'status': 'settled' is removed because ALL historical markets are settled
    }
    if cursor:
        params["cursor"] = cursor

    # CHANGE 1: Point to /historical/markets instead of /markets
    resp = requests.get(f"{base}/historical/markets", params=params).json()
    
    # CHANGE 2: Manually filter the returned markets by your 1-year window
    markets = resp.get("markets", [])
    for market in markets:
        close_time_str = market.get("close_time")
        if close_time_str:
            close_dt = parser.parse(close_time_str)
            # Only keep the market if it falls in your window
            if start_date <= close_dt <= end_date:
                all_markets.append(market)
    
    cursor = resp.get("cursor")
    if not cursor:
        break

print(f"Total historical markets found: {len(all_markets)}")

Total historical markets found: 4414
